# Task 3: Interactive Plotly Report [Hard]
 Using plotly express, recreate 3 of your matplotlib charts as interactive equivalents. 

 Add custom hover data (at least 2 columns shown on hover), color-encode a categorical variable, and export each as a standalone HTML file. 
 
 Additionally, build a 4th interactive chart type not covered in class (e.g., choropleth map, sunburst, treemap) with documented code explaining your design choices.

### Import Libraries

In [2]:
import pandas as pd
import plotly.express as px

### Load Dataset

In [3]:
df = pd.read_csv(r"..\..\week_4(ETL_Data_Cleaning)\Task_8\tvmaze_cleaned.csv")

df.head()

,id,url,name,type,language,genres,status,premiered,ended,summary,rating_average,schedule_time,schedule_days,premier_year,rating_category,summary_word_count,show_duration_years
0,1,https://www.tvmaze.com/shows/1/under-the-dome,Under the Dome,Scripted,English,"['Drama', 'Science-Fiction', 'Thriller']",Ended,2013-06-24,2015-09-10,Under the Dome is the story of a small town th...,6.6,22:00,Thursday,2013,Good,57,2.0
1,2,https://www.tvmaze.com/shows/2/person-of-interest,Person of Interest,Scripted,English,"['Action', 'Crime', 'Science-Fiction']",Ended,2011-09-22,2016-06-21,You are being watched. The government has a se...,8.8,22:00,Tuesday,2011,Excellent,96,5.0
2,3,https://www.tvmaze.com/shows/3/bitten,Bitten,Scripted,English,"['Drama', 'Horror', 'Romance']",Ended,2014-01-11,2016-04-15,Based on the critically acclaimed series of no...,7.4,22:00,Friday,2014,Good,75,2.0
3,4,https://www.tvmaze.com/shows/4/arrow,Arrow,Scripted,English,"['Drama', 'Action', 'Science-Fiction']",Ended,2012-10-10,2020-01-28,"After a violent shipwreck, billionaire playboy...",7.4,21:00,Tuesday,2012,Good,85,8.0
4,5,https://www.tvmaze.com/shows/5/true-detective,True Detective,Scripted,English,"['Drama', 'Crime', 'Thriller']",Running,2014-01-12,NaN,Touch darkness and darkness touches you back. ...,8.1,21:00,Sunday,2014,Excellent,47,12.0


### Basic Cleaning

In [4]:
# Check dataset info
print(df.info())

# Check missing values
print(df.isnull().sum())

# Remove rows where rating is missing
df = df.dropna(subset=["rating_average"])

# Convert date columns to datetime
df["premiered"] = pd.to_datetime(df["premiered"], errors="coerce")
df["ended"] = pd.to_datetime(df["ended"], errors="coerce")

# Remove duplicate rows if any
df = df.drop_duplicates()

# Clean text columns
df["name"] = df["name"].str.strip()
df["language"] = df["language"].str.strip()
df["type"] = df["type"].str.strip()

# Ensure numerical columns are proper numeric types
numeric_cols = [
    "rating_average",
    "premier_year",
    "summary_word_count",
    "show_duration_years"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove rows with invalid years
df = df[df["premier_year"] > 1900]

# Final shape after cleaning
print("Cleaned Dataset Shape:", df.shape)

<class 'pandas.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   240 non-null    int64  
 1   url                  240 non-null    str    
 2   name                 240 non-null    str    
 3   type                 240 non-null    str    
 4   language             240 non-null    str    
 5   genres               240 non-null    str    
 6   status               240 non-null    str    
 7   premiered            240 non-null    str    
 8   ended                219 non-null    str    
 9   summary              240 non-null    str    
 10  rating_average       240 non-null    float64
 11  schedule_time        225 non-null    str    
 12  schedule_days        235 non-null    str    
 13  premier_year         240 non-null    int64  
 14  rating_category      240 non-null    str    
 15  summary_word_count   240 non-null    int64  
 16  s

### CHART 1 — INTERACTIVE BAR CHART

Average Rating by Show Type

In [7]:
avg_rating = (
    df.groupby("type", as_index=False)["rating_average"]
    .mean()
)

fig1 = px.bar(
    avg_rating,
    x="type",
    y="rating_average",
    color="type",
    title="Average Rating by Show Type",
    
    hover_data={
        "type": True,
        "rating_average": ':.2f'
    }
)

fig1.update_layout(
    xaxis_title="Show Type",
    yaxis_title="Average Rating"
)

fig1.write_html("interactive_bar_chart.html")

fig1.show()

### CHART 2 — INTERACTIVE SCATTER PLOT

Duration vs Rating

In [8]:
fig2 = px.scatter(
    df,
    x="show_duration_years",
    y="rating_average",
    color="type",

    hover_data=[
        "name",
        "language",
        "summary_word_count"
    ],

    title="Show Duration vs Rating"
)

fig2.update_layout(
    xaxis_title="Duration (Years)",
    yaxis_title="Average Rating"
)

fig2.write_html("interactive_scatter_plot.html")

fig2.show()

### CHART 3 — INTERACTIVE LINE CHART

Shows Released Over Time

In [9]:
year_data = (
    df["premier_year"]
    .value_counts()
    .sort_index()
    .reset_index()
)

year_data.columns = ["premier_year", "show_count"]

fig3 = px.line(
    year_data,
    x="premier_year",
    y="show_count",
    markers=True,

    hover_data={
        "premier_year": True,
        "show_count": True
    },

    title="TV Show Releases Over Time"
)

fig3.update_layout(
    xaxis_title="Year",
    yaxis_title="Number of Shows"
)

fig3.write_html("interactive_line_chart.html")

fig3.show()

### CHART 4 — TREEMAP (NEW CHART TYPE)

Hierarchical Distribution of Shows

In [10]:
fig4 = px.treemap(
    df,
    path=["type", "rating_category"],
    values="summary_word_count",
    color="rating_average",

    hover_data=[
        "language",
        "show_duration_years"
    ],

    title="TV Show Type and Rating Category Treemap"
)

fig4.write_html("interactive_treemap.html")

fig4.show()

## Treemap Design Choices

A treemap was selected to visualize hierarchical relationships between TV show types and rating categories. 
The chart groups shows first by type and then by rating category, allowing viewers to quickly compare category sizes and rating performance. 
Color intensity represents average ratings, while rectangle size reflects total summary word count, helping highlight which categories dominate the dataset.